# Embedding Workflow

In [ ]:
import openai
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
client = openai.AsyncOpenAI()

In [ ]:
response = await client.responses.create(
    model='gpt-4o-mini',
    input='hi'
)

In [ ]:
response.output_text

In [ ]:
phrases = [
    'hello', 'hi', 'good-bye', 'see ya later', 'moose', '1 + 1 = 2', '2 + 2 = 5', 'qperqoweirupqweor',
    '!@#$%^&*()_', 'def foobar(): return 7', 'specificity', 'agent engineering',
    'Utah'
]

response = await client.embeddings.create(
    input=phrases,
    model='text-embedding-3-small'
)

In [ ]:
embeds = np.array([emb.embedding for emb in response.data])

In [ ]:
embeds

In [ ]:
async def embed(text: str) -> np.array:
    response = await client.embeddings.create(
        input=[text],
        model='text-embedding-3-small'
    )
    return np.array(response.data[0].embedding)

In [ ]:
phrase = 'hola'
query = await embed(phrase)

In [ ]:
ax = plt.bar(x=range(len(phrases)), height=(query @ embeds.T))
plt.xticks(range(len(phrases)), phrases, rotation=45, ha='right', rotation_mode='anchor')
plt.title('Embedding similiarity for ' + phrase)
plt.ylim([0, 1]);

## Pipeline

In [ ]:
async def embed(content: list[str]) -> np.array:
    response = await client.embeddings.create(
        input=content,
        model='text-embedding-3-small'
    )
    return np.array([emb.embedding for emb in response.data])

In [ ]:
from pathlib import Path

In [ ]:
content_dir = Path('../../../cs301R/gospel-connections/data/text/1-ne/')

In [ ]:
content_verses = []
for content_file in sorted(content_dir.glob('*.txt'), key=lambda f: f.name):
    content_verses += content_file.read_text().splitlines()

In [ ]:
content_verses

In [ ]:
content_embeds = await embed(content_verses)

In [ ]:
content_embeds

In [ ]:
async def get_verses(phrase, threshold=0.6):
    embedding = await embed([phrase])
    scores = content_embeds @ embedding.T
    return np.array(content_verses)[scores.flatten() > threshold]

In [ ]:
hits = await get_verses('build boat', threshold=0.32)

In [ ]:
for hit in hits:
    print(hit)
    print()